In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.metrics import r2_score
import pandas as pd
import json
import numpy as np
import pickle
import random
from torch.cuda.amp import GradScaler, autocast
import csv
import warnings


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def trusted_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:

        return torch.load(path, map_location=map_location)


class SimpleGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)


        if hasattr(self, 'edge_norm') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)

        u = getattr(data, 'u', None)
        if hasattr(self, 'global_norm') and u is not None:
            u = self.global_norm(u)

        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out


class EnhancedGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None

        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)

        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)

        if self.edge_norm and hasattr(data, 'edge_attr'):
            _ = self.edge_norm(data.edge_attr)

        u = getattr(data, 'u', None)
        if u is not None:
            u = self.global_norm(u)
            gf = self.global_mlp(u)

        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        pooled = global_mean_pool(x, data.batch)
        h = torch.cat([pooled, gf], dim=1) if u is not None else pooled
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out


class GateNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_teachers):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_teachers)

    def forward(self, h):
        a = F.relu(self.fc1(h))
        return F.softmax(self.fc2(a), dim=-1)


class Adapter(nn.Module):
    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)

    def forward(self, h):
        return self.linear(h)


def create_data_loader(graph_list, batch_size=32, shuffle=True):
    data_list = []
    for g in graph_list:
        data_list.append(Data(
            x=g['x'],
            edge_index=g['edge_index'],
            edge_attr=g.get('edge_attr', None),
            u=g.get('u', None),
            y=g['y'],
            y_soft=g.get('y_soft', None)
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)


LOSS_ABLATIONS = {
    'hard_only': {
        'label': 'Existing hard term only',
        'use_soft': False,
        'use_feature': False,
    },
    'hard_soft': {
        'label': 'Existing hard + soft terms',
        'use_soft': True,
        'use_feature': False,
    },
    'hard_feature': {
        'label': 'Existing hard + feature terms',
        'use_soft': False,
        'use_feature': True,
    },
    'hard_soft_feature': {
        'label': 'Existing hard + soft + feature terms',
        'use_soft': True,
        'use_feature': True,
    },
}


EXPECTED_TEACHER_ORDER = ('qcut', 'elem', 'molwt', 'fp', 'scaffold')


def teacher_name_from_path(path):
    filename = os.path.basename(path).lower()
    for name in EXPECTED_TEACHER_ORDER:
        if name in filename:
            return name
    return 'unknown'


def validate_teacher_order(teacher_paths):
    detected = tuple(teacher_name_from_path(p) for p in teacher_paths)
    if detected != EXPECTED_TEACHER_ORDER:
        raise ValueError(
            "Teacher checkpoint order does not match the soft-label columns.\n"
            f"detected={detected}\n"
            f"expected={EXPECTED_TEACHER_ORDER}"
        )
    return detected


def train_model(
    train_dir,
    val_dir,
    teacher_paths,
    save_path,
    ablation_mode,
    gate_hidden=128,
    hint_lambda=5.0,
    weight_ratio=(0.6, 0.4),
    hidden_dims=[128, 128],
    dropout=0.1,
    epochs=500,
    batch_size=64,
    lr=1e-3,
    min_lr=1e-4,
    lr_patience=20,
    es_patience=50,
    seed=42,
):
    if ablation_mode not in LOSS_ABLATIONS:
        raise ValueError(
            f"Unknown ablation_mode={ablation_mode}; "
            f"Valid options={list(LOSS_ABLATIONS.keys())}"
        )

    mode_cfg = LOSS_ABLATIONS[ablation_mode]
    use_soft = mode_cfg['use_soft']
    use_feature = mode_cfg['use_feature']


    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(
        f"\n[{ablation_mode}] seed={seed} | device={device} | "
        f"hint_lambda={hint_lambda} | weight_ratio={weight_ratio} | "
        f"gate_hidden={gate_hidden}"
    )


    train_g = trusted_torch_load(os.path.join(train_dir, 'graph_data.pt'))
    val_g = trusted_torch_load(os.path.join(val_dir, 'graph_data.pt'))


    ys = torch.stack([g['y'] for g in train_g]).view(-1)
    y_mean = ys.mean().item()
    y_std = ys.std().item() + 1e-8

    for g in train_g:
        g['y'] = (g['y'] - y_mean) / y_std

        g.setdefault('y_soft', g['y'])

    for g in val_g:
        g['y'] = (g['y'] - y_mean) / y_std

        g.setdefault('y_soft', g['y'])


    tr_loader = create_data_loader(train_g, batch_size, True)
    va_loader = create_data_loader(val_g, batch_size, False)


    sample = train_g[0]
    n_dim = sample['x'].size(1)
    e_dim = (
        sample.get('edge_attr', None).size(1)
        if sample.get('edge_attr', None) is not None
        else 0
    )
    g_dim = (
        sample.get('u', None).size(1)
        if sample.get('u', None) is not None
        else 0
    )

    student = EnhancedGNN(
        n_dim, e_dim, g_dim, hidden_dims, dropout
    ).to(device)


    teacher_names = validate_teacher_order(teacher_paths)

    teachers = []
    teacher_load_report = []

    for p in teacher_paths:
        ck = trusted_torch_load(p, map_location=device)

        t = SimpleGNN(
            ck['node_dim'],
            ck.get('edge_dim', 0),
            ck.get('global_dim', 0),
            ck['hidden_dims'],
            ck['dropout']
        ).to(device)


        incompatible = t.load_state_dict(
            ck['model_state_dict'],
            strict=False
        )

        teacher_load_report.append({
            'path': p,
            'missing_keys': list(incompatible.missing_keys),
            'unexpected_keys': list(incompatible.unexpected_keys),
        })


        t.eval()
        teachers.append(t)

    print(f"Loaded {len(teachers)} teachers: {teacher_names}")

    for report in teacher_load_report:
        if report['missing_keys'] or report['unexpected_keys']:
            print(
                "[WARNING] Teacher checkpoint strict=False mismatch:\n"
                f"  {os.path.basename(report['path'])}\n"
                f"  missing={report['missing_keys']}\n"
                f"  unexpected={report['unexpected_keys']}"
            )


    K = len(teachers)
    gate = GateNet(student.final_dim, gate_hidden, K).to(device)
    adapter = Adapter(student.final_dim, student.final_dim).to(device)

    optimizer = optim.Adam(
        list(student.parameters()) +
        list(gate.parameters()) +
        list(adapter.parameters()),
        lr=lr,
        weight_decay=1e-5
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=lr_patience,
        min_lr=min_lr
    )

    scaler = GradScaler(enabled=(device.type == 'cuda'))

    best_r2 = -1e9
    patience = 0

    history = {
        'loss': [],
        'hard_term': [],
        'soft_term': [],
        'feature_term': [],
        'train_r2': [],
        'val_r2': [],
        'lr': [],
    }


    def eval_loader(loader):
        student.eval()
        yy, pp = [], []

        with torch.no_grad():
            for b in loader:
                b = b.to(device)
                out, _ = student(b, return_feat=True)
                yy.append(b.y.view(-1).cpu().numpy())
                pp.append(np.atleast_1d(out.detach().cpu().numpy()))

        return r2_score(
            np.concatenate(yy),
            np.concatenate(pp)
        )

    hard_weight, soft_weight = weight_ratio


    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = 0.0
        total_hard = 0.0
        total_soft = 0.0
        total_feature = 0.0

        for batch in tr_loader:
            batch = batch.to(device)

            with autocast(enabled=(device.type == 'cuda')):
                pred_s, h_s = student(batch, return_feat=True)


                Ht = torch.stack(
                    [
                        t(batch, return_feat=True)[1]
                        for t in teachers
                    ],
                    dim=1
                )


                w = gate(h_s)


                Ht_g = (
                    w.unsqueeze(-1) * Ht
                ).sum(dim=1)

                loss_hint = F.mse_loss(
                    adapter(h_s),
                    Ht_g
                )


                fused = (
                    w * batch.y_soft
                ).sum(dim=1)


                pred_f = (
                    hard_weight * pred_s +
                    soft_weight * fused
                )


                hard_term = (
                    hard_weight *
                    F.mse_loss(
                        pred_f,
                        batch.y.view(-1)
                    )
                )

                soft_term = (
                    soft_weight *
                    F.mse_loss(
                        pred_s,
                        fused
                    )
                )

                feature_term = (
                    hint_lambda *
                    loss_hint
                )


                loss = hard_term

                if use_soft:
                    loss = loss + soft_term

                if use_feature:
                    loss = loss + feature_term

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()

            total_loss += float(loss.item())
            total_hard += float(hard_term.item())
            total_soft += float(soft_term.item())
            total_feature += float(feature_term.item())


        train_r2 = eval_loader(tr_loader)
        val_r2 = eval_loader(va_loader)

        avg_loss = total_loss / len(tr_loader)
        avg_hard = total_hard / len(tr_loader)
        avg_soft = total_soft / len(tr_loader)
        avg_feature = total_feature / len(tr_loader)
        lr_now = optimizer.param_groups[0]['lr']

        history['loss'].append(avg_loss)
        history['hard_term'].append(avg_hard)
        history['soft_term'].append(avg_soft)
        history['feature_term'].append(avg_feature)
        history['train_r2'].append(train_r2)
        history['val_r2'].append(val_r2)
        history['lr'].append(lr_now)

        if epoch % 30 == 0:
            print(
                f"[{ablation_mode}] Epoch {epoch} | "
                f"Loss {avg_loss:.4f} | "
                f"Hard {avg_hard:.4f} | "
                f"Soft(raw) {avg_soft:.4f} | "
                f"Feature(raw) {avg_feature:.4f} | "
                f"Train R2 {train_r2:.4f} | "
                f"Val R2 {val_r2:.4f} | "
                f"LR {lr_now:.1e}"
            )

        scheduler.step(avg_loss)

        if val_r2 > best_r2:
            best_r2 = val_r2
            patience = 0

            torch.save({
                'model_state_dict': student.state_dict(),
                'gate_state': gate.state_dict(),
                'adapter_state': adapter.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'history': history,
                'node_dim': n_dim,
                'edge_dim': e_dim,
                'global_dim': g_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout,


                'ablation_mode': ablation_mode,
                'use_soft': use_soft,
                'use_feature': use_feature,
                'legacy_exact': True,
                'seed': seed,
                'gate_hidden': gate_hidden,
                'hint_lambda': hint_lambda,
                'weight_ratio': weight_ratio,
                'teacher_order': teacher_names,
                'teacher_load_report': teacher_load_report,
                'best_val_r2': best_r2,
            }, save_path)

            if epoch % 30 != 0:
                print(
                    f"[{ablation_mode}] Saved best model "
                    f"at Epoch {epoch} | Val R2={best_r2:.6f}"
                )

        else:
            patience += 1
            if patience >= es_patience:
                print(
                    f"[{ablation_mode}] Early stopping "
                    f"at epoch {epoch}"
                )
                break

    print(
        f"[{ablation_mode}] Training complete | "
        f"best Val R2={best_r2:.10f}"
    )


    return save_path


if __name__ == '__main__':
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    project_root = os.path.abspath(os.getcwd())
    config_path = os.path.join(project_root, 'config', 'loss_ablation_seed_hyperparameters.csv')


    seed_configs = {}
    with open(config_path, newline='', encoding='utf-8') as config_file:
        for row in csv.DictReader(config_file):
            seed = int(row['seed'])
            if seed in seed_configs:
                raise ValueError(f'Duplicate parameter entry for seed {seed}.')
            seed_configs[seed] = row
    if set(seed_configs) != set(seeds):
        raise ValueError('The configuration CSV must contain exactly the ten specified seeds.')
    missing = [seed for seed in seeds if any(
        not seed_configs[seed][field]
        for field in ('hint_lambda', 'weight_student', 'weight_teacher', 'gate_hidden', 'config_id')
    )]
    if missing:
        raise ValueError(f'Verified seed-specific parameters are required for seeds: {missing}.')

    train_dir = os.path.join(project_root, 'data-set', 'train')
    val_dir = os.path.join(project_root, 'data-set', 'validation')
    teacher_order = ('qcut', 'elem', 'molwt', 'fp', 'scaffold')
    teacher_dir = os.path.join(project_root, 'checkpoints', 'teachers')
    teacher_paths = [os.path.join(teacher_dir, f'{name}.pt') for name in teacher_order]
    save_root = os.path.join(project_root, 'results', 'loss_ablation')
    os.makedirs(save_root, exist_ok=True)

    fixed_config = {
        'hidden_dims': [128, 128], 'dropout': 0.1,
        'epochs': 1000, 'batch_size': 64, 'lr': 1e-3,
        'min_lr': 5e-5, 'lr_patience': 30, 'es_patience': 100,
    }
    ablation_modes = ('hard_soft_feature', 'hard_only', 'hard_soft', 'hard_feature')
    manifest_path = os.path.join(save_root, 'checkpoint_manifest.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as manifest_file:
        writer = csv.DictWriter(manifest_file, fieldnames=['seed', 'ablation_mode', 'config_id', 'checkpoint_path'])
        writer.writeheader()
        for seed in seeds:
            row = seed_configs[seed]
            hint_lambda = float(row['hint_lambda'])
            weights = (float(row['weight_student']), float(row['weight_teacher']))
            gate_hidden = int(row['gate_hidden'])
            if hint_lambda < 0 or gate_hidden <= 0 or any(weight < 0 for weight in weights) or abs(sum(weights) - 1.0) > 1e-6:
                raise ValueError(f'Invalid fixed hyperparameters for seed {seed}.')
            for ablation_mode in ablation_modes:
                mode_dir = os.path.join(save_root, ablation_mode)
                os.makedirs(mode_dir, exist_ok=True)
                checkpoint_path = os.path.join(
                    mode_dir, f"student_{ablation_mode}_{row['config_id']}_seed{seed}.pt"
                )
                train_model(
                    train_dir=train_dir, val_dir=val_dir, teacher_paths=teacher_paths,
                    save_path=checkpoint_path, ablation_mode=ablation_mode,
                    gate_hidden=gate_hidden, hint_lambda=hint_lambda,
                    weight_ratio=weights, seed=seed, **fixed_config,
                )
                writer.writerow({
                    'seed': seed, 'ablation_mode': ablation_mode,
                    'config_id': row['config_id'],
                    'checkpoint_path': os.path.relpath(checkpoint_path, project_root),
                })
                manifest_file.flush()
    print(f'Loss ablation completed. Checkpoint manifest: {manifest_path}')
